In [1]:
import os
import re
import sys
import glob
import time
import uproot
import ternary
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime

from scipy.signal import savgol_filter

from functools import partial

from matplotlib.colors import LogNorm

from collections import Counter

In [2]:
### Import functions from peak_functions

repository_path = os.path.abspath('/Users/romoluque_c/Repositories/BACoN_romo/')
sys.path.append(repository_path)

import blr_functions  as blr
import peak_functions as pf
import fit_functions  as fitf

In [3]:
import matplotlib

# Set Palatino as the serif font
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif']  = 'Palatino'

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
plt.rcParams["figure.figsize"] = 8, 5
plt.rcParams["font.size"]      = 14

In [6]:
fav_col = '#418094'

## Analyze npz data

In [15]:
def extract_date_obj_and_number(filename):
    match       = re.search(r"(\d{2}_\d{2}_\d{4})-file_(\d+)", filename)
    date_str    = match.group(1)
    file_number = int(match.group(2))
    date_obj    = datetime.strptime(date_str, "%m_%d_%Y")
    return date_obj, file_number

def extract_date_and_number(filename):
    match = re.search(r"(\d{2}_\d{2}_\d{4})-file_(\d+)", filename)
    return match.group(1), int(match.group(2))

def adapt_evt_id(evt_id, fnum):
    return 1000000*fnum + np.array(evt_id)

thr_ADC        = 90
sg_filt_window = 30
min_dist       = 25

normal_chs  = range(9)
trigger_chs = [9, 10, 11]
pmt_ch      = 12

evts_path    = '/Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/'
out_path     = '/Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz_cuts/'
filt_files   = glob.glob(evts_path + f'/BACoN_run6_hits_and_times_newint_thr{thr_ADC}_mean_w{sg_filt_window}_dist{min_dist}_run-04_1*.npz')
sorted_files = sorted(filt_files, key=extract_date_obj_and_number)
len(sorted_files)

941

In [16]:
def expected_pmt_file(evts_path, date, fnum):
    return f"{evts_path}/BACoN_pmt_analysis_peaks_thr60_dist15_run-{date}-file_{fnum}.npz"

def outfile_name(out_path, text, date, fnum):
    return f"{out_path}/BACoN_{text}_run-{date}-file_{fnum}.npz"

## Cut 1: events that saturate in the PMT

In [25]:
start_time  = time.time()
pmt_sat_thr = 6000 #15000
for ifile, sipm_file in enumerate(sorted_files):

    d = np.load(sipm_file, allow_pickle=True)
    date, fnum = extract_date_and_number(sipm_file)
    
    pmt_file = expected_pmt_file(evts_path,             date, fnum)
    outfile  = outfile_name(     out_path, 'filt_cut2', date, fnum)
    
    if not os.path.exists(pmt_file):
        print(f"Skipping: PMT file missing for {date} {fnum}")
        continue
    if ifile % 20 == 0:
        print(f"Processing file {ifile}: {sipm_file}")

    pmt_data  = np.load(pmt_file,  allow_pickle=True)

    d_idx_max_pmt = pmt_data['idx_peaks_pmt']
    h_peaks_pmt   = pmt_data['h_peaks_pmt']
    max_h_pmt     = np.array([np.max(el) if np.size(el) > 0 else 0 for el in h_peaks_pmt])
    sat_evts_pmt  = np.where(max_h_pmt>pmt_sat_thr)[0]
    
    tot_evts_file    = len(d_idx_max_pmt)
    tot_sat_evts_pmt = len(sat_evts_pmt)
    
    np.savez(outfile,
             sat_evts_pmt    =sat_evts_pmt,
             tot_evts_file   =tot_evts_file,
             tot_sat_evts_pmt=tot_sat_evts_pmt)
    
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} s, {elapsed_time/60} min")

Processing file 0: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_0.npz
Processing file 20: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_20.npz
Processing file 40: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_40.npz
Processing file 60: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_60.npz
Processing file 80: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_80.npz
Processing file 100: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_100.npz
Processing file 120: /Users/romoluque_c/LEGEND/BACON

## Cut 2: use the ternary plots to select events

In [18]:
import json

#with open("/Users/romoluque_c/Repositories/BACON_romo/example_notebooks/Run6/SiPM_calibration_using_integral.json", "r") as f:
#    cal_i_all = json.load(f)
with open("/Users/romoluque_c/Repositories/BACON_romo/example_notebooks/Run6/SiPM_calibration_trigg_using_height.json", "r") as f:
    cal_h_trigg = json.load(f)
with open("/Users/romoluque_c/Repositories/BACON_romo/example_notebooks/Run6/SiPM_calibration_using_height.json", "r") as f:
    cal_h_non_trigg = json.load(f)

In [20]:
cal_h_trigg, cal_h_non_trigg

({'04_10_2026': {'9': [951.3753523168559, -37.30307894600901],
   '10': [777.2288506214306, 26.00755244572838],
   '11': [872.5143197476924, 27.987995700048486]},
  '04_13_2026': {'9': [950.9217906448679, -36.11773638566865],
   '10': [777.048629662605, 25.772030115466805],
   '11': [871.3637655350273, 31.22257313017917]},
  '04_16_2026': {'9': [952.1921066911822, -35.95434259862759],
   '10': [777.1795896538729, 26.16051491641383],
   '11': [871.1546517780267, 30.15856326062203]},
  '04_18_2026': {'9': [952.1330039466678, -36.49215782252031],
   '10': [777.1739290564144, 25.9183961327235],
   '11': [871.692626241088, 29.646660781842456]}},
 {'0': [181.7658173473375, 1.4702445485033877],
  '1': [141.65333476654476, 0.23709673432676936],
  '2': [169.69603785333467, -0.414230688157204],
  '3': [175.90477360936532, -0.6843470718768003],
  '4': [195.5819111173151, -0.1747241493196725],
  '5': [179.34075649766888, 0.07176771719605313],
  '6': [148.56628534116314, 0.15593608309200135],
  '7'

In [21]:
def get_sum_of_light_for_given_ranges(data, lo_lim=0.25, hi_lim=0.50):
    total     = np.sum(data[ch]       for ch in trigger_chs)
    fractions = {ch: data[ch] / total for ch in trigger_chs}
    
    mask = np.ones_like(total, dtype=bool)
    for ch in trigger_chs:
        mask &= (fractions[ch] >= lo_lim) & (fractions[ch] <= hi_lim)
    
    selected = {ch: data[ch][mask] for ch in trigger_chs}
    indcs    = np.where(mask==True)[0]
    return indcs, selected

In [22]:
start_time = time.time()
print(len(sorted_files))
for ifile, sipm_file in enumerate(sorted_files):
    if ifile % 20 == 0:
        print(f"Processing file {ifile}: {sipm_file}")

    d = np.load(sipm_file, allow_pickle=True)
    date, fnum = extract_date_and_number(sipm_file)
    outfile    = outfile_name(out_path, 'filt_cut3', date, fnum)
    h_sg_trigg = d['height_peaks_sg_trigg'].item()
    
    ## Transform to PE
    h_sg_trigg_PE = {ch: (np.array(h_sg_trigg[ch]) - cal_h_trigg[date][str(ch)][1]) / cal_h_trigg[date][str(ch)][0]
                     for ch in trigger_chs}
    
    sum_heights_ch_PE       = {ch: np.fromiter(map(np.sum, h_sg_trigg_PE[ch]), dtype=float) for ch in trigger_chs}
    sum_h_PE_indcs_10_75, _ = get_sum_of_light_for_given_ranges(sum_heights_ch_PE, lo_lim=0.10, hi_lim=0.75)
    sum_h_PE_indcs_20_60, _ = get_sum_of_light_for_given_ranges(sum_heights_ch_PE, lo_lim=0.20, hi_lim=0.60)
    sum_h_PE_indcs_25_50, _ = get_sum_of_light_for_given_ranges(sum_heights_ch_PE, lo_lim=0.25, hi_lim=0.50)
    sum_h_PE_indcs_30_40, _ = get_sum_of_light_for_given_ranges(sum_heights_ch_PE, lo_lim=0.30, hi_lim=0.40)
    
    tot_evts_file       = len(sum_heights_ch_PE[9])
    tot_evts_h_PE_10_75 = len(sum_h_PE_indcs_10_75)
    tot_evts_h_PE_20_60 = len(sum_h_PE_indcs_20_60)
    tot_evts_h_PE_25_50 = len(sum_h_PE_indcs_25_50)
    tot_evts_h_PE_30_40 = len(sum_h_PE_indcs_30_40)

    np.savez(outfile,
             tot_evts_file       =tot_evts_file,
             tot_evts_h_PE_10_75 =tot_evts_h_PE_10_75,
             tot_evts_h_PE_20_60 =tot_evts_h_PE_20_60,
             tot_evts_h_PE_25_50 =tot_evts_h_PE_25_50,
             tot_evts_h_PE_30_40 =tot_evts_h_PE_30_40,
             sum_h_PE_indcs_10_75=sum_h_PE_indcs_10_75,
             sum_h_PE_indcs_20_60=sum_h_PE_indcs_20_60,
             sum_h_PE_indcs_25_50=sum_h_PE_indcs_25_50,
             sum_h_PE_indcs_30_40=sum_h_PE_indcs_30_40)
    
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} s, {elapsed_time/60} min")

941
Processing file 0: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_0.npz
Processing file 20: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_20.npz
Processing file 40: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_40.npz
Processing file 60: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_60.npz
Processing file 80: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_80.npz
Processing file 100: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_100.npz
Processing file 120: /Users/romoluque_c/LEGEND/B

## Cut 3: Pretrigger cut

In [23]:
def filter_evts_by_trigg_time(indcs_array, ampls_array, idx_thr=650, amp_thr=0.5):

    n_events   = len(indcs_array)
    bad_events = np.zeros(n_events, dtype=bool)

    for i, (indcs, ampls) in enumerate(zip(indcs_array, ampls_array)):
        if np.any((indcs < idx_thr) & (ampls > amp_thr)):
            bad_events[i] = True

    good_events_idx = np.where(~bad_events)[0]

    return good_events_idx

In [24]:
start_time = time.time()
for ifile, sipm_file in enumerate(sorted_files):

    d = np.load(sipm_file, allow_pickle=True)
    date, fnum = extract_date_and_number(sipm_file)
    
    pmt_file = expected_pmt_file(evts_path,             date, fnum)
    outfile  = outfile_name(     out_path, 'filt_cut5', date, fnum)
    
    # --- skip events without PMT file ---
    if not os.path.exists(pmt_file):
        print(f"Skipping: PMT file missing for {date} {fnum}")
        continue
    if ifile % 20 == 0:
        print(f"Processing file {ifile}: {sipm_file}")

    sipm_data = np.load(sipm_file, allow_pickle=True)
    pmt_data  = np.load(pmt_file,  allow_pickle=True)
    
    d_filt_evts     = sipm_data['filt_evts_dict']     .item()
    d_idx_max       = sipm_data['idx_peaks_max']      .item()
    d_idx_max_trigg = sipm_data['idx_peaks_max_trigg'].item()
    d_idx_max_pmt   = pmt_data ['idx_peaks_pmt']
    
    h_sg        = sipm_data['height_peaks_sg']      .item()
    h_sg_trigg  = sipm_data['height_peaks_sg_trigg'].item()
    h_peaks_pmt = pmt_data ['h_peaks_pmt']
    
    h_sg_PE = {ch: (np.array(h_sg[ch]) - cal_h_non_trigg[str(ch)][1]) / cal_h_non_trigg[str(ch)][0]
               for ch in normal_chs}
    h_sg_trigg_PE = {ch: (np.array(h_sg_trigg[ch]) - cal_h_trigg[date][str(ch)][1]) / cal_h_trigg[date][str(ch)][0]
                     for ch in trigger_chs}

    
    k_idcs = {ch: filter_evts_by_trigg_time(d_idx_max[ch], h_sg_PE[ch]) for ch in normal_chs}
    f_evts = {ch: [d_filt_evts[ch][i] for i in k_idcs[ch]] for ch in normal_chs}
    
    f_evts_trigg = {ch: filter_evts_by_trigg_time(d_idx_max_trigg[ch], h_sg_trigg_PE[ch]) for ch in trigger_chs}
    f_evts_pmt   =      filter_evts_by_trigg_time(d_idx_max_pmt, h_peaks_pmt, amp_thr=60)
    
    # Count events passing cuts
    tot_evts_file             =          len(d_idx_max_pmt)
    tot_evts_pretrigg_t_n_chs = {ch:     len(f_evts      [ch]) for ch in normal_chs}
    tot_evts_pretrigg_t_trigg = {ch:     len(f_evts_trigg[ch]) for ch in trigger_chs}
    tot_evts_pretrigg_t_pmt   = {pmt_ch: len(f_evts_pmt)}
    
    tot_evts_pretrigg_t = {**tot_evts_pretrigg_t_n_chs,
                           **tot_evts_pretrigg_t_trigg,
                           **tot_evts_pretrigg_t_pmt}
    
    np.savez(outfile,
             f_evts       =f_evts,
             f_evts_trigg =f_evts_trigg,
             f_evts_pmt   =f_evts_pmt,
             tot_evts_file=tot_evts_file,
             tot_evts_pretrigg_t=tot_evts_pretrigg_t)
    
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} s, {elapsed_time/60} min")

Processing file 0: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_0.npz
Processing file 20: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_20.npz
Processing file 40: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_40.npz
Processing file 60: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_60.npz
Processing file 80: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_80.npz
Processing file 100: /Users/romoluque_c/LEGEND/BACON/new_setup_run6/data_npz/BACoN_run6_hits_and_times_newint_thr90_mean_w30_dist25_run-04_10_2026-file_100.npz
Processing file 120: /Users/romoluque_c/LEGEND/BACON